# TP 1 - Ingénierie de consignes modèle


---
## 0. Configuration partagée


In [1]:
import json

from shared.config import ROOT_DIR
from shared.llm_utils import LLMRequest, run_llm
from shared.misc_utils import (
    find_and_parse_json_from_text,
    write_json_file,
)

LOG_DIR = ROOT_DIR / "TP1_travel_planner_LLM" / "logs"

user_query = """
Je veux partir 4 jours à Rome en avril, je n'ai pas encore les dates exactes.
Propose-moi un itinéraire de voyage. Mon budget est de 200 euros pour les sorties et les restaurants.
Je veux éviter les zones trop touristiques et découvrir des lieux plus confidentiels.
"""

#### Fonctions utilitaires pour estimer le coût et afficher l'usage des tokens

In [2]:
def token_estimate_cost_usd(
    token_usage: dict[str, int],
    input_price_per_1m_tokens_usd: float = 0.30,
    output_price_per_1m_tokens_usd: float = 2.50,
) -> float:
    input_cost = token_usage["input_tokens"] * input_price_per_1m_tokens_usd / 1_000_000
    output_cost = token_usage["output_tokens"] * output_price_per_1m_tokens_usd / 1_000_000
    return input_cost + output_cost


def token_print_report(token_usage: dict[str, int]) -> None:
    estimated_cost = token_estimate_cost_usd(token_usage)
    print(
        f"tokens : entrée={token_usage['input_tokens']} | "
        f"sortie={token_usage['output_tokens']} | "
        f"total={token_usage['total_tokens']}"
    )
    print(f"coût estimé (USD) : {estimated_cost:.6f}")


---
### Au préalable

On prépare une base technique pour la logique d'appel LLM

- `project_settings` : configuration partagée du modèle (température, top_p, top_k, max_tokens, budget de réflexion)
- `genai_client` : client Google GenAI authentifié, créé une seule fois
- `LLMRequest` **(TODO)** : classe qui représente les données d'entrée d'un appel LLM (`user_prompt` obligatoire, `system_prompt` optionnel)
- `LLMResponse` **(TODO)** : classe qui représente les données de sortie utiles (texte final, tokens, données brutes)
- `run_llm` **(TODO)** : fonction qui lit la configuration, envoie la requête au modèle et retourne un `LLMResponse`

Les fonctions et classes marquées TODO sont à implémenter dans `shared/llm_utils.py`.

---
## 1. Version 1: requête utilisateur seule


**TODO — Version V1**

Fichier à modifier : `TP1_travel_planner_LLM/1_llm_assistant.ipynb`


Bloc de code qui construit la requête minimale avec `LLMRequest`, appelle `run_llm`, puis enregistre la sortie et les tokens dans `logs/llm_output_v1.txt`


In [3]:
# TODO : construire la requête minimale V1 sans system prompt
request_v1 = LLMRequest(
    system_prompt="",
    user_prompt="Aide moi à planifier un voyage de 4 jours à Rome en avril avec un budget de 200 euros pour les sorties et les restaurants, en évitant les zones trop touristiques."
)
run_result_v1 = await run_llm(request_v1)
final_text_v1 = run_result_v1.output

token_usage_v1 = {
    "input_tokens": run_result_v1.input_tokens,
    "output_tokens": run_result_v1.output_tokens,
    "total_tokens": run_result_v1.total_tokens,
}

log_path_v1 = LOG_DIR / "llm_output_v1.txt"
write_json_file(file_path=log_path_v1, data=run_result_v1.raw_response)

print(final_text_v1)

Absolument ! Planifier un voyage à Rome avec un budget serré et en évitant les foules, c'est tout à fait possible et même très gratifiant. Avril est une excellente période, le temps est généralement agréable et les fleurs sont magnifiques. Voici une proposition de plan pour 4 jours, axée sur les expériences authentiques et abordables.

**Philosophie du voyage :**

*   **Se déplacer intelligemment :** Prioriser la marche et les transports en commun (bus et métro) pour économiser.
*   **Manger comme un local :** Privilégier les trattorias familiales, les marchés, et les "pizza al taglio" (pizza à la coupe).
*   **Découvrir la Rome "cachée" :** Explorer les quartiers moins connus, les jardins secrets, et les églises moins fréquentées mais tout aussi magnifiques.
*   **Profiter des gratuités :** Rome regorge de sites gratuits ou à prix très abordables.

**Budget (200€ pour 4 jours pour les sorties et restaurants) :**

Cela représente environ 50€ par jour. C'est un budget serré mais réalisa

In [6]:
token_print_report(token_usage_v1)


tokens : entrée=44 | sortie=1024 | total=1068
coût estimé (USD) : 0.002573


---
## 2. Version 2: prompt système structuré


**TODO — Version V2**

Fichier à modifier : `TP1_travel_planner_LLM/1_llm_assistant.ipynb`

L'objectif est d'améliorer la qualité de la réponse avec des instructions claires.<br>
Pour cela, il faut définir :
- **Rôle** : ...
- **Contraintes** : ...
- **Structure** : 4 sections — résumé, itinéraire, budget, conseils.
- **Notes additionnelles** : *utilise cette section pour toute précision ou règle spéciale.*

**But :**
- Produire une réponse complète.
- Rester sous 3000 tokens.


In [10]:
# TODO : rédiger un system prompt contraint et réutilisable
system_prompt_v2 = """
## Instructions
Tu es un asssistant de voyage spécialisé dans la planification de voyages à Rome. Tu es franc et concis.

Contraintes à respecter:
- Moins de 1000 tokens dans ta réponse
- ne pas segmenter par période de la journée (matin, après-midi, soir) mais plutôt par jour entier

Contraintes de style:
- Formel 
- Concis
- Markdown

Structure de réponse obligatoire:
1) Enumère les critères demandés par l'utilisateur dans une section "Demande" sous forme de liste à puces
2) Propose un itinéraire de voyage structuré jour par jour dans une section "Itinéraire" sous forme de liste numérotée, avec pour chaque jour une liste à puces des activités recommandées.
Pour chaque activité, indique une estimation du coût en euros.
3) Section "Budget" : indique le budget total estimé pour les activités proposées. Une seule valeur chiffrée en euros, sans texte explicatif.
4) Section "Sources" : liste les sources utilisées pour construire l'itinéraire

...
"""

request_v2 = LLMRequest(
    system_prompt=system_prompt_v2,
    user_prompt=user_query,
)
run_result_v2 = await run_llm(request_v2)
final_text_v2 = run_result_v2.output

token_usage_v2 = {
    "input_tokens": int(run_result_v2.input_tokens),
    "output_tokens": int(run_result_v2.output_tokens),
    "total_tokens": int(run_result_v2.total_tokens),
}

log_path_v2 = LOG_DIR / "llm_output_v2.txt"
write_json_file(file_path=log_path_v2, data=run_result_v2.raw_response)

print(final_text_v2)

## Demande

*   Durée du séjour : 4 jours
*   Période : Avril
*   Budget pour sorties et restaurants : 200 euros
*   Préférence : Éviter les zones trop touristiques, découvrir des lieux confidentiels.

## Itinéraire

**Jour 1 : Trastevere et Janicule**

*   Balade dans le quartier de Trastevere, exploration des ruelles moins fréquentées.
*   Visite de la Basilique Sainte-Marie-du-Trastevere (Gratuit).
*   Montée au Gianicolo (Janicule) pour une vue panoramique sur la ville, loin des foules du Capitole.
*   Dîner dans une trattoria typique à Trastevere (Budget restaurant : 30€).

**Jour 2 : Quartier Monti et Célio**

*   Exploration du quartier Monti, connu pour ses boutiques d'artisans et ses places cachées.
*   Visite de la Basilique Saint-Clément-du-Latran, avec ses niveaux souterrains fascinants (Entrée : 10€).
*   Découverte du quartier du Célio, visite du parc de la Villa Celimontana (Gratuit).
*   Dîner dans le quartier Monti (Budget restaurant : 30€).

**Jour 3 : Aventin et Test

In [11]:
token_print_report(token_usage_v2)


tokens : entrée=310 | sortie=709 | total=1019
coût estimé (USD) : 0.001865


---
## 3. Version 3: sortie structurée


**TODO — Version V3**

Fichier à modifier : `TP1_travel_planner_LLM/1_llm_assistant.ipynb`

L'objectif ici est d'avoir une sortie structurée, facile à parser et à traiter.
- Pour cela, il faut imposer un schéma strict JSON (ou XML, ou TOON), en plus de la réponse texte.
- Le prompt système final sera : <br>`system_prompt_v3 = system_prompt_v2 + structured_output_instructions`
- Il faut aussi créer une fonction de parsing pour extraire et parser le bloc JSON depuis le texte final.
- Enfin, on itérera sur ce JSON pour mesurer le coût total estimé et le comparer à la contrainte de budget.


In [15]:
# TODO : imposer la section JSON finale avec schéma strict
system_prompt_v3 = system_prompt_v2 + """

### Format de réponse
produit un JSON avec la structure suivante :
{demande: [markdown de la partie "Demande"], itineraire: [markdown de la partie "Itinéraire"], budget: [int de la valeur dans "Budget"], days: [int des jours dans "Demande"], sources: [markdown de la partie "Sources"]}
"""

request_v3 = LLMRequest(
    system_prompt=system_prompt_v3,
    user_prompt=user_query,
)
run_result_v3 = await run_llm(request_v3)
final_text_v3 = run_result_v3.output

token_usage_v3 = {
    "input_tokens": int(run_result_v3.input_tokens),
    "output_tokens": int(run_result_v3.output_tokens),
    "total_tokens": int(run_result_v3.total_tokens),
}

log_path_v3 = LOG_DIR / "llm_output_v3.txt"
write_json_file(file_path=log_path_v3, data=run_result_v3.raw_response)

print(final_text_v3)

```json
{
  "demande": "- Durée : 4 jours\n- Période : Avril\n- Budget : 200 euros (sorties et restaurants)\n- Préférence : Éviter les zones trop touristiques, découvrir des lieux confidentiels",
  "itineraire": "## Itinéraire\n\n**Jour 1**\n*   **Quartier du Trastevere :** Flâner dans les ruelles pavées, visiter la Basilique Santa Maria in Trastevere.\n    *   Coût estimé : 0€ (visite basilique gratuite)\n*   **Jardin des Orangers (Giardino degli Aranci) :** Profiter d'une vue panoramique sur la ville.\n    *   Coût estimé : 0€\n*   **Bocca della Verità :** Une curiosité romaine.\n    *   Coût estimé : 0€\n*   **Dîner dans une trattoria typique du Trastevere.**\n    *   Coût estimé : 30€\n\n**Jour 2**\n*   **Quartier Monti :** Explorer les boutiques d'artisans, les petites places.\n    *   Coût estimé : 0€\n*   **Marché de Monti (si ouvert le jour de votre visite) :** Chercher des créations locales.\n    *   Coût estimé : 0€ (achat facultatif)\n*   **Thermes de Caracalla :** Découvrir

In [16]:
token_print_report(token_usage_v3)


tokens : entrée=385 | sortie=880 | total=1265
coût estimé (USD) : 0.002316


#### Parser la sortie structurée

**TODO — `find_and_parse_json_from_text`**

Fichier à modifier : `shared/misc_utils.py`

`find_and_parse_json_from_text` : fonction qui récupère le dernier bloc JSON valide dans un texte de réponse, puis retourne un dictionnaire Python ou lève une erreur explicite


In [17]:
# TODO : parser la sortie texte+JSON sans post-traitement manuel
parsed_json_v3 = find_and_parse_json_from_text(final_text_v3)
print(parsed_json_v3)

{'demande': '- Durée : 4 jours\n- Période : Avril\n- Budget : 200 euros (sorties et restaurants)\n- Préférence : Éviter les zones trop touristiques, découvrir des lieux confidentiels', 'itineraire': "## Itinéraire\n\n**Jour 1**\n*   **Quartier du Trastevere :** Flâner dans les ruelles pavées, visiter la Basilique Santa Maria in Trastevere.\n    *   Coût estimé : 0€ (visite basilique gratuite)\n*   **Jardin des Orangers (Giardino degli Aranci) :** Profiter d'une vue panoramique sur la ville.\n    *   Coût estimé : 0€\n*   **Bocca della Verità :** Une curiosité romaine.\n    *   Coût estimé : 0€\n*   **Dîner dans une trattoria typique du Trastevere.**\n    *   Coût estimé : 30€\n\n**Jour 2**\n*   **Quartier Monti :** Explorer les boutiques d'artisans, les petites places.\n    *   Coût estimé : 0€\n*   **Marché de Monti (si ouvert le jour de votre visite) :** Chercher des créations locales.\n    *   Coût estimé : 0€ (achat facultatif)\n*   **Thermes de Caracalla :** Découvrir les vestiges

#### Vérification du budget

Finalement, on peut traiter le JSON parsé pour calculer le coût total estimé des activités proposées, et vérifier que cela respecte la contrainte de budget donnée dans la requête utilisateur.

In [18]:


total_cost_eur = parsed_json_v3["budget"]
average_per_day = total_cost_eur / parsed_json_v3["days"]

print(f"Total estimé : {total_cost_eur:.2f} EUR")
print(f"Moyenne par jour : {average_per_day:.2f} EUR")


Total estimé : 120.00 EUR
Moyenne par jour : 30.00 EUR
